In [21]:
import time
import math
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import FinanceDataReader as fdr
import pymysql
from DATA.stock_invest_function import get_db_host   # 사용자 환경에 이미 존재한다고 하셨음
from typing import Union, Optional, List, Dict, Any

# =========================================================
# 0) 설정
# =========================================================
DB_NAME = "investar"  # 필요 시 변경
TABLE_RESULT = "us_required_return_result"
INDICATOR_COL = "indicator"

MARKET_TICKER = "SPY"   # price_mkt 용
MAX_RETRY = 5
SLEEP_BETWEEN_CALLS = 0.25  # 5000 tickers면 너무 짧게 잡지 마세요(0.2~1.0 권장)

# beta window
BETA_WINDOWS = {
    "beta_252": 252,
    "beta_750": 750,
    "beta_1250": 1250
}

# required return horizons (trailing window for E[Rm])
HORIZONS = {
    "1y": 252,
    "3y": 252*3,
    "5y": 252*5
}


def _get_json(url: str) -> Union[List[Any], Dict[str, Any]]:
    for k in range(MAX_RETRY):
        r = requests.get(url, timeout=30)
        if r.status_code == 429:
            time.sleep(1.0 + 0.5 * k)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError("FMP 429 too many retries")

# 1) DB 설정
# db_info = {
#     "host": get_db_host(),
#     "port": 3307,
#     "user": "stox7412",
#     "password": "Apt106503!~",   # 실제 비밀번호
#     "database": "investar",
# }


# =========================================================
# 1) DB 유틸
# =========================================================
def get_conn():
    host = get_db_host()
    return pymysql.connect(
        host=host,
        port=3307,
        user = "stox7412",           # 본인 환경에 맞게
        password = "Apt106503!~",    # 본인 환경에 맞게
        db=DB_NAME,
        charset="utf8mb4",
        autocommit=False,
        cursorclass=pymysql.cursors.DictCursor
    )

def ensure_result_table():
    """
    스샷 기준 테이블 구조가 이미 존재하므로,
    보통은 CREATE TABLE을 하지 않는 것을 권장합니다.

    다만, 정말 테이블이 없을 때만 만들고 싶다면 아래처럼 'indicator'로 생성해야 합니다.
    """
    sql = f"""
    CREATE TABLE IF NOT EXISTS {TABLE_RESULT} (
        date DATE NOT NULL,
        ticker VARCHAR(20) NOT NULL,
        indicator VARCHAR(50) NOT NULL,
        value DOUBLE NULL,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
        PRIMARY KEY (date, ticker, indicator),
        INDEX idx_ticker_indicator (ticker, indicator),
        INDEX idx_indicator_date (indicator, date)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
    finally:
        conn.close()

def read_price_from_db(ticker: str) -> pd.DataFrame:
    """
    DB long-format 테이블에서 indicator='price_stock' 읽어서 (date, price_stock)로 반환
    """
    sql = f"""
    SELECT date, value AS price_stock
    FROM {TABLE_RESULT}
    WHERE ticker=%s AND {INDICATOR_COL}='price_stock'
    ORDER BY date;
    """
    conn = get_conn()
    try:
        df = pd.read_sql(sql, conn, params=[ticker])
    finally:
        conn.close()

    if df.empty:
        return df

    df["date"] = pd.to_datetime(df["date"])
    df = df.dropna(subset=["price_stock"]).drop_duplicates("date")
    return df

def get_last_date_in_db(ticker: str) -> Optional[pd.Timestamp]:
    sql = f"""
    SELECT MAX(date) AS last_date
    FROM {TABLE_RESULT}
    WHERE ticker=%s AND {INDICATOR_COL}='price_stock';
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (ticker,))
            row = cur.fetchone()
            last_date = row["last_date"] if row else None
    finally:
        conn.close()

    if last_date is None:
        return None
    return pd.to_datetime(last_date)

def upsert_long(df_long: pd.DataFrame, batch_rows: int = 5000):
    if df_long is None or df_long.empty:
        return

    df = df_long.copy()

    # 1) 필수 컬럼 존재 확인
    required_cols = ["date", "ticker", "indicator", "value"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"upsert_long: missing columns {missing}")

    # 2) date 정리 (NaT 제거)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"].notna()]
    df["date"] = df["date"].dt.date

    # 3) ticker/indicator는 NaN이면 안 됨
    df["ticker"] = df["ticker"].astype(str)
    df["indicator"] = df["indicator"].astype(str)
    df = df[(df["ticker"].notna()) & (df["ticker"] != "nan") & (df["ticker"] != "None")]
    df = df[(df["indicator"].notna()) & (df["indicator"] != "nan") & (df["indicator"] != "None")]

    # 4) value: NaN/inf를 확실히 None으로 변환
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df.loc[~np.isfinite(df["value"]), "value"] = np.nan  # inf/-inf -> nan
    # pandas NaN -> 파이썬 None으로 강제 변환 (DB NULL)
    df["value"] = df["value"].astype(object)
    df.loc[df["value"].isna(), "value"] = None

    # 5) 혹시라도 남아있는 NaN 방어 (최종)
    # (object dtype에서도 np.nan이 남을 수 있어서 tuple 생성 전에 한번 더)
    def _clean_value(x):
        if x is None:
            return None
        try:
            if isinstance(x, float) and (math.isnan(x) or math.isinf(x)):
                return None
        except Exception:
            pass
        return float(x)

    df["value"] = df["value"].apply(_clean_value)

    insert_sql = f"""
    INSERT INTO {TABLE_RESULT} (date, ticker, indicator, value)
    VALUES (%s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        value=VALUES(value),
        updated_at=CURRENT_TIMESTAMP;
    """

    # 6) rows 생성 (여기서 NaN이 있으면 다시 터짐 → 위에서 완전 제거)
    rows = list(df[["date", "ticker", "indicator", "value"]].itertuples(index=False, name=None))

    conn = get_conn()
    try:
        with conn.cursor() as cur:
            for i in range(0, len(rows), batch_rows):
                cur.executemany(insert_sql, rows[i:i+batch_rows])
            conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()



# =========================================================
# 2) FMP 호출 (DB 이후 구간만)
# =========================================================
# def _get_json(url: str) -> list | dict:
#     for k in range(MAX_RETRY):
#         r = requests.get(url, timeout=30)
#         if r.status_code == 429:
#             time.sleep(1.0 + 0.5*k)
#             continue
#         r.raise_for_status()
#         return r.json()
#     raise RuntimeError("FMP 429 too many retries")

def fetch_fmp_price(symbol: str, api_key: str, start_date: str) -> pd.DataFrame:
    """
    FMP historical-price-full: start_date 이후만 가져오고, Adj Close를 우선 사용.
    """
    url = (
        f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
        f"?from={start_date}&apikey={api_key}"
    )
    js = _get_json(url)
    hist = js.get("historical", []) if isinstance(js, dict) else []
    if not hist:
        return pd.DataFrame(columns=["date", "price"])

    df = pd.DataFrame(hist)
    # FMP는 date 내림차순이 많아서 정렬
    df["date"] = pd.to_datetime(df["date"])
    price_col = "adjClose" if "adjClose" in df.columns else "close"
    df = df[["date", price_col]].rename(columns={price_col: "price"})
    df = df.dropna().drop_duplicates("date").sort_values("date")
    return df

def fetch_fmp_treasury(api_key: str) -> pd.DataFrame:
    """
    FMP treasury: maturity별 금리(%)를 받는 endpoint가 계정 플랜에 따라 다를 수 있습니다.
    아래는 /api/v4/treasury 형태를 가정.
    """
    url = f"https://financialmodelingprep.com/api/v4/treasury?apikey={api_key}"
    js = _get_json(url)
    df = pd.DataFrame(js) if isinstance(js, list) else pd.DataFrame()
    if df.empty:
        return df

    # 기대 컬럼 예시: date, year1, year2, year3, year5, year10 ...
    df["date"] = pd.to_datetime(df["date"])
    # 1Y/3Y/5Y만 사용
    keep = []
    for c in df.columns:
        lc = c.lower()
        if lc in ["date", "year1", "year3", "year5", "1year", "3year", "5year"]:
            keep.append(c)
    df = df[keep].sort_values("date").drop_duplicates("date")
    # 표준 컬럼명으로 정리
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc in ["year1", "1year"]:
            rename[c] = "rf_1y"
        if lc in ["year3", "3year"]:
            rename[c] = "rf_3y"
        if lc in ["year5", "5year"]:
            rename[c] = "rf_5y"
    df = df.rename(columns=rename)
    # 단위: % -> decimal
    for col in ["rf_1y", "rf_3y", "rf_5y"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce") / 100.0
    return df


# =========================================================
# 3) 계산 로직 (Beta + Required Return)
# =========================================================
def calc_returns(price: pd.Series) -> pd.Series:
    return price.pct_change()

def rolling_beta(ret_stock: pd.Series, ret_mkt: pd.Series, window: int) -> pd.Series:
    """
    beta = Cov(rs, rm)/Var(rm) rolling
    """
    cov = ret_stock.rolling(window).cov(ret_mkt)
    var = ret_mkt.rolling(window).var()
    beta = cov / var
    return beta

def trailing_ann_geometric_return(daily_ret: pd.Series, window: int) -> pd.Series:
    """
    trailing window의 기하평균 연환산:
    (Π(1+r))^(252/window) - 1
    """
    def f(x):
        x = x.dropna()
        if len(x) < max(30, int(window*0.6)):  # 너무 짧으면 NaN
            return np.nan
        g = np.prod(1.0 + x.values)
        if g <= 0:
            return np.nan
        return g ** (252.0 / len(x)) - 1.0

    return daily_ret.rolling(window).apply(f, raw=False)

def build_features(price_stock_df: pd.DataFrame,
                   price_mkt_df: pd.DataFrame,
                   rf_df: pd.DataFrame) -> pd.DataFrame:
    """
    입력:
      price_stock_df: columns [date, price_stock]
      price_mkt_df:   columns [date, price_mkt]
      rf_df:          columns [date, rf_1y, rf_3y, rf_5y] (빈 경우 허용)

    출력:
      date index, features columns
    """
    df = pd.merge(price_stock_df, price_mkt_df, on="date", how="inner")
    df = df.sort_values("date").drop_duplicates("date")
    df["ret_stock"] = calc_returns(df["price_stock"])
    df["ret_mkt"] = calc_returns(df["price_mkt"])

    # beta windows
    for name, w in BETA_WINDOWS.items():
        df[name] = rolling_beta(df["ret_stock"], df["ret_mkt"], w)

    # rf 붙이기 (가장 최근 공시값 forward-fill)
    if rf_df is not None and not rf_df.empty:
        df = pd.merge_asof(
            df.sort_values("date"),
            rf_df.sort_values("date"),
            on="date",
            direction="backward"
        )
    else:
        df["rf_1y"] = np.nan
        df["rf_3y"] = np.nan
        df["rf_5y"] = np.nan

    # E[Rm] horizons (market)
    df["E_Rm_1y"] = trailing_ann_geometric_return(df["ret_mkt"], HORIZONS["1y"])
    df["E_Rm_3y"] = trailing_ann_geometric_return(df["ret_mkt"], HORIZONS["3y"])
    df["E_Rm_5y"] = trailing_ann_geometric_return(df["ret_mkt"], HORIZONS["5y"])

    # Required return: Re = rf + beta_252*(E[Rm]-rf)  (beta 기준은 252로 통일)
    # 원하시면 horizon별로 beta_750/beta_1250를 쓰도록 바꿀 수 있습니다.
    beta_base = df["beta_252"]

    df["Re_1y"] = df["rf_1y"] + beta_base * (df["E_Rm_1y"] - df["rf_1y"])
    df["Re_3y"] = df["rf_3y"] + beta_base * (df["E_Rm_3y"] - df["rf_3y"])
    df["Re_5y"] = df["rf_5y"] + beta_base * (df["E_Rm_5y"] - df["rf_5y"])

    return df


# =========================================================
# 4) 메인 파이프라인 - Python 3.8/3.9/3.10 호환
# =========================================================
def update_one_ticker(ticker: str, api_key: str, today: Optional[str] = None) -> None:
    """
    1) DB에서 price_stock 읽기
    2) 마지막 날짜 이후만 FMP에서 보충
    3) SPY도 동일 로직으로 보충(또는 통째로 FMP에서 필요한 구간만)
    4) feature 계산
    5) long format upsert
    """
    if today is None:
        today = datetime.utcnow().date().isoformat()

    price_db = read_price_from_db(ticker)
    last_dt = get_last_date_in_db(ticker)

    if last_dt is None:
        start_date = (pd.to_datetime(today) - pd.tseries.offsets.BDay(252 * 6)).date().isoformat()
        price_new = fetch_fmp_price(ticker, api_key, start_date=start_date)
        price_stock = price_new.rename(columns={"price": "price_stock"})
    else:
        start_missing = (last_dt + pd.Timedelta(days=1)).date().isoformat()
        if start_missing <= today:
            price_new = fetch_fmp_price(ticker, api_key, start_date=start_missing)
            time.sleep(SLEEP_BETWEEN_CALLS)
        else:
            price_new = pd.DataFrame(columns=["date", "price"])

        price_new = price_new.rename(columns={"price": "price_stock"})
        price_stock = pd.concat([price_db, price_new], ignore_index=True)
        price_stock = price_stock.dropna().drop_duplicates("date").sort_values("date")

    if price_stock.empty:
        print(f"[WARN] {ticker}: price_stock 데이터가 없습니다.")
        return

    start_for_mkt = price_stock["date"].min().date().isoformat()

    mkt = fetch_fmp_price(MARKET_TICKER, api_key, start_date=start_for_mkt)
    time.sleep(SLEEP_BETWEEN_CALLS)
    mkt = mkt.rename(columns={"price": "price_mkt"})
    if mkt.empty:
        print(f"[WARN] market({MARKET_TICKER}) price 없음. {ticker} 스킵")
        return

    rf = fetch_fmp_treasury(api_key)
    time.sleep(SLEEP_BETWEEN_CALLS)

    feat = build_features(price_stock_df=price_stock, price_mkt_df=mkt, rf_df=rf)
    feat = feat.dropna(subset=["price_stock", "price_mkt"])
    feat["ticker"] = ticker

    keep_cols = [
        "price_stock", "price_mkt",
        "ret_stock", "ret_mkt",
        "beta_252", "beta_750", "beta_1250",
        "rf_1y", "rf_3y", "rf_5y",
        "E_Rm_1y", "E_Rm_3y", "E_Rm_5y",
        "Re_1y", "Re_3y", "Re_5y"
    ]
    keep_cols = [c for c in keep_cols if c in feat.columns]

    df_long = feat.melt(
        id_vars=["date", "ticker"],
        value_vars=keep_cols,
        var_name=INDICATOR_COL,   # ★ item -> indicator
        value_name="value"
    )

    upsert_long(df_long)
    print(f"[OK] {ticker}: {len(df_long):,} rows upsert")


def run_batch(tickers: List[str], api_key: str, start_idx: int = 0) -> None:
    ensure_result_table()
    for i, t in enumerate(tickers[start_idx:], start=start_idx):
        try:
            update_one_ticker(t, api_key)
        except Exception as e:
            print(f"[ERROR] {t} 실패: {e}")


def get_filtered_us_tickers() -> List[str]:
    """
    NASDAQ / NYSE / AMEX 전체 상장사 정보를 기반으로
    특정 IndustryCode 앞 4자리에 해당하는 기업들을 제외하고
    최종 ticker 리스트를 반환하는 함수.
    """
    nasdaq = fdr.StockListing('NASDAQ')
    nyse = fdr.StockListing('NYSE')
    amex = fdr.StockListing('AMEX')

    info_df = pd.concat([nasdaq, nyse, amex], ignore_index=True)

    exclude_prefixes = ["5510", "5730", "5530", "6010", "5910", "5120"]

    info_df["IndustryCode"] = info_df["IndustryCode"].astype(str)
    info_df["IndustryPrefix"] = info_df["IndustryCode"].str[:4]

    mask_exclude = info_df["IndustryPrefix"].isin(exclude_prefixes)

    excluded_df = info_df[mask_exclude]
    filtered_df = info_df[~mask_exclude].copy()

    tickers = filtered_df["Symbol"].unique().tolist()

    print(f"제외된 기업 수: {len(excluded_df)}")
    print(f"남은 기업 수: {len(filtered_df)}")
    print(f"티커 수: {len(tickers)}")

    return tickers


# =========================================================
# 실행 예시
# =========================================================
# if __name__ == "__main__":
#     API_KEY = "YOUR_FMP_KEY"
#
#     # 예시
#     tickers = ["AAPL", "MSFT", "NVDA"]
#     run_batch(tickers, API_KEY)


In [22]:
# 1) DB 설정
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

tickers  = ["AAPL", "MSFT", "NVDA"]  # 예시

# tickers = get_filtered_us_tickers()

In [23]:
run_batch(tickers, API_KEY)

[ERROR] AAPL 실패: nan can not be used with MySQL
[ERROR] MSFT 실패: nan can not be used with MySQL
[ERROR] NVDA 실패: nan can not be used with MySQL


In [24]:
def get_unique_indicators(limit: int = None) -> pd.DataFrame:
    """
    us_required_return_result 테이블에 존재하는
    indicator의 unique 값과 개수를 조회
    """
    sql = f"""
    SELECT indicator, COUNT(*) AS cnt
    FROM {TABLE_RESULT}
    GROUP BY indicator
    ORDER BY cnt DESC
    """

    if limit is not None:
        sql += f"\nLIMIT {int(limit)}"

    conn = get_conn()
    try:
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    return df


In [25]:
# 실행
indicator_df = get_unique_indicators()
print(indicator_df)

   indicator  cnt
0  indicator  cnt
1  indicator  cnt
2  indicator  cnt
3  indicator  cnt
4  indicator  cnt
5  indicator  cnt


In [26]:
def check_header_pollution(db_info, table="us_required_return_result"):
    conn = pymysql.connect(
        host=db_info["host"], port=db_info.get("port", 3306),
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4"
    )
    try:
        # 1) 의심 행 개수(대표적인 헤더 문자열)
        sql = f"""
        SELECT
            SUM(ticker IN ('ticker','Ticker','TICKER')) AS bad_ticker_rows,
            SUM(indicator IN ('indicator','Indicator','INDICATOR','cnt','CNT')) AS bad_indicator_rows,
            SUM(CAST(date AS CHAR) IN ('date','Date','DATE')) AS bad_date_rows
        FROM {table};
        """
        df1 = pd.read_sql(sql, conn)

        # 2) 실제 샘플 몇 줄 보기
        sql2 = f"""
        SELECT date, ticker, indicator, value
        FROM {table}
        WHERE ticker IN ('ticker','Ticker','TICKER')
           OR indicator IN ('indicator','Indicator','INDICATOR','cnt','CNT')
           OR CAST(date AS CHAR) IN ('date','Date','DATE')
        LIMIT 50;
        """
        df2 = pd.read_sql(sql2, conn)

        return df1, df2
    finally:
        conn.close()

summary_df, sample_bad_df = check_header_pollution(db_info)
print(summary_df)
print(sample_bad_df)


   bad_ticker_rows  bad_indicator_rows  bad_date_rows
0              0.0                 0.0            0.0
Empty DataFrame
Columns: [date, ticker, indicator, value]
Index: []


In [27]:
def debug_unique_indicators_raw(db_info, table="us_required_return_result", limit=30):
    conn = pymysql.connect(
        host=db_info["host"], port=db_info.get("port", 3306),
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor
    )
    try:
        sql = f"""
        SELECT indicator, COUNT(*) AS cnt
        FROM {table}
        GROUP BY indicator
        ORDER BY cnt DESC
        LIMIT {int(limit)};
        """
        with conn.cursor() as cur:
            cur.execute(sql)
            rows = cur.fetchall()
        print("raw rows sample:")
        for r in rows[:10]:
            print(r)
        return rows
    finally:
        conn.close()

rows = debug_unique_indicators_raw(db_info, table="us_required_return_result", limit=50)

raw rows sample:
{'indicator': 'Re_1y', 'cnt': 3216823}
{'indicator': 'beta_252', 'cnt': 3216823}
{'indicator': 'Re_3y', 'cnt': 2589524}
{'indicator': 'beta_750', 'cnt': 2589524}
{'indicator': 'Re_5y', 'cnt': 2010111}
{'indicator': 'beta_1250', 'cnt': 2010111}


In [28]:
def check_price_indicators(db_info, table="us_required_return_result"):
    candidates = ["price_stock", "price", "close", "adjClose", "adj_close", "Close", "Adj Close"]
    conn = pymysql.connect(
        host=db_info["host"], port=db_info.get("port", 3306),
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor
    )
    try:
        with conn.cursor() as cur:
            out = []
            for name in candidates:
                cur.execute(f"SELECT COUNT(*) AS cnt FROM {table} WHERE indicator=%s;", (name,))
                out.append({"indicator": name, "cnt": cur.fetchone()["cnt"]})
        return pd.DataFrame(out).sort_values("cnt", ascending=False)
    finally:
        conn.close()

print(check_price_indicators(db_info))

     indicator  cnt
0  price_stock    0
1        price    0
2        close    0
3     adjClose    0
4    adj_close    0
5        Close    0
6    Adj Close    0
